# Class-mixture posterior means and ensembles of pretrained diffusion priors

`diffusion-tweedie-benchmark` found that the one-step posterior mean of the frozen ADM 256×256 unconditional model
reaches DMID-d's published state of the art, and that symmetry ensembles add only ~0.013 dB: flips of one network
make correlated errors. `adm-tweedie-finetune` found that fine-tuning on DIV2K at the denoising steps adds ≤ 0.01 dB.

This run asks whether *different* pretrained networks help, without any training:

* **class mixture:** E[x | y] = Σ_c p(c | y) E[x | y, c], with p(c | x_t) from OpenAI's noise-aware ImageNet
  classifier and E[x | x_t, c] from the class-conditional ADM (Tweedie at the matched step), over the top-3 classes;
  for the 256×256 and the 512×512 conditional models;
* **equal-weight ensembles** of the unconditional, conditional and fine-tuned networks' one-step estimates.

Protocol as in `diffusion-tweedie-benchmark` (DMID's noise, the same seeds, whole image reflect-padded to 64).
Kaggle: GPU T4 ×2, internet on; attach the output of `adm-tweedie-finetune` for the fine-tuned member (optional).

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lpips"], check=True)

In [ ]:
import glob, io, json, math, os, queue, re, subprocess, sys, threading, time, traceback, urllib.request, zipfile
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image

TEST_MODE = os.environ.get("TEST_MODE") == "1"     # CPU smoke test: tiny random networks, small crops
SIGMAS = [15, 25, 50]
DATASETS = ["McMaster", "Kodak24", "CBSD68"]       # same order (and seeds) as diffusion-tweedie-benchmark
TOP_K = 3
TIME_LIMIT_H = 11.0
DEVICES = [torch.device(f"cuda:{i}") for i in range(torch.cuda.device_count())] or [torch.device("cpu")]
assert TEST_MODE or DEVICES[0].type == "cuda", "GPU required"
torch.backends.cudnn.benchmark = False
CACHE, OUT = os.environ.get("CACHE_DIR", "/tmp/cache"), "outputs"
INPUT_ROOTS = ["/kaggle/input", os.environ.get("INPUT_DIR", "/nonexistent")]
for d in (CACHE, OUT):
    os.makedirs(d, exist_ok=True)
T_START, TIMES = time.time(), {}
print([str(d) for d in DEVICES], torch.__version__, "TEST_MODE" if TEST_MODE else "")


def download(url, path, size=None):
    """Resumable download with progress; no-op when the file is already complete."""
    have = os.path.getsize(path) if os.path.exists(path) else 0
    if have and (size is None or have == size):
        return path
    if size and have > size:
        have = 0
    req = urllib.request.Request(url, headers={"Range": f"bytes={have}-"} if have else {})
    with urllib.request.urlopen(req, timeout=60) as r:
        if have and r.status != 206:
            have = 0  # server ignored the Range header: start over
        total, t0, got, mark = have + int(r.headers.get("Content-Length", 0)), time.time(), 0, 0
        with open(path, "ab" if have else "wb") as f:
            while chunk := r.read(1 << 23):
                f.write(chunk)
                got += len(chunk)
                if got - mark >= 1 << 30:
                    mark = got
                    print(f"  {path.split('/')[-1]}: {(have + got) / 1e9:.1f}/{total / 1e9:.1f} GB  {got / 1e6 / (time.time() - t0):.0f} MB/s", flush=True)
    if size:
        assert os.path.getsize(path) == size, f"incomplete download: {path}"
    return path


def natkey(s):
    return [int(t) if t.isdigit() else t for t in re.split(r"(\d+)", s)]


t0 = time.time()
OPENAI = "https://openaipublic.blob.core.windows.net/diffusion/jul-2021/"
FILES = {
    "uncond": (OPENAI + "256x256_diffusion_uncond.pt", 2_211_383_297),
    "cond256": (OPENAI + "256x256_diffusion.pt", 2_215_479_544),
    "cls256": (OPENAI + "256x256_classifier.pt", 216_496_432),
    "cond512": (OPENAI + "512x512_diffusion.pt", 2_236_135_873),
    "cls512": (OPENAI + "512x512_classifier.pt", 217_919_056),
    "datasets": ("https://github.com/Li-Tong-621/DMID/releases/download/v1.0/Datasets.zip", 367_018_356),
}
if TEST_MODE:
    FILES = {"datasets": FILES["datasets"]}
with ThreadPoolExecutor(6) as ex:
    P = dict(zip(FILES, ex.map(lambda kv: download(kv[1][0], f"{CACHE}/{kv[0]}{os.path.splitext(kv[1][0])[1]}", kv[1][1]), FILES.items())))
GD = f"{CACHE}/guided-diffusion"
if not os.path.isdir(GD):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/openai/guided-diffusion", GD], check=True)
FT_PATH = next((p for root in INPUT_ROOTS for p in sorted(glob.glob(f"{root}/**/adm_ft_ema.pt", recursive=True))), None)
TIMES["downloads"] = time.time() - t0
print(f"downloads {TIMES['downloads'] / 60:.1f} min; fine-tuned weights: {FT_PATH}")

## 1 · Networks (fp16; ADM's attention through `scaled_dot_product_attention` on contiguous q, k, v)

In [ ]:
sys.path[:0] = [GD]
from guided_diffusion.script_util import create_classifier, create_model
from guided_diffusion import unet as gd_unet


def sdpa_attention(self, qkv):
    bs, width, length = qkv.shape
    ch = width // (3 * self.n_heads)
    q, k, v = (z.transpose(1, 2).contiguous() for z in qkv.reshape(bs * self.n_heads, ch * 3, length).split(ch, dim=1))
    return F.scaled_dot_product_attention(q, k, v).transpose(1, 2).reshape(bs, -1, length)


gd_unet.QKVAttentionLegacy.forward = sdpa_attention
gd_unet.QKVAttention.forward = lambda self, qkv: sdpa_attention(self, qkv)  # same arithmetic for the new attention order
FLAGS = dict(num_channels=256, num_res_blocks=2, learn_sigma=True, attention_resolutions="32,16,8", num_head_channels=64,
             resblock_updown=True, use_scale_shift_norm=True)
CLS_FLAGS = dict(classifier_width=128, classifier_depth=2, classifier_attention_resolutions="32,16,8",
                 classifier_use_scale_shift_norm=True, classifier_resblock_updown=True, classifier_pool="attention")
if TEST_MODE:
    FLAGS.update(num_channels=32, num_head_channels=16, num_res_blocks=1)   # matches the fine-tune test's tiny weights
    CLS_FLAGS.update(classifier_width=64, classifier_depth=1)              # 512's channel_mult starts at 0.5: 32 groups need ≥ 64
FP16 = not TEST_MODE
STATES = {k: torch.load(P[k], map_location="cpu", weights_only=True) for k in ("uncond", "cond256", "cond512", "cls256", "cls512") if k in P}
if FT_PATH:
    STATES["ft"] = {k: v.float() for k, v in torch.load(FT_PATH, map_location="cpu", weights_only=True).items()}


def make(kind, device):
    torch.manual_seed(0)
    if kind.startswith("cls"):
        m = create_classifier(image_size=int(kind[3:]), classifier_use_fp16=FP16, **CLS_FLAGS)
    else:
        size = 512 if kind == "cond512" else 256
        flags = dict(FLAGS, num_channels=64) if TEST_MODE and size == 512 else FLAGS   # 512's channel_mult starts at 0.5
        m = create_model(image_size=size, class_cond=kind.startswith("cond"), use_fp16=FP16, **flags)
    if kind in STATES:
        m.load_state_dict(STATES[kind])
    m.to(device).eval().requires_grad_(False)
    if FP16:
        m.convert_to_fp16()
    return m


import lpips

KINDS = ["uncond", "cond256", "cond512", "cls256", "cls512"] + (["ft"] if FT_PATH else [])   # "ft": unconditional architecture
CTX = [dict(dev=d, lpips=lpips.LPIPS(net="alex", verbose=False).to(d), **{k: make(k, d) for k in KINDS}) for d in DEVICES]
STATES.clear()
print("networks:", KINDS)

## 2 · Estimators

In [ ]:
BETAS = torch.linspace(1e-4, 0.02, 1000, dtype=torch.float64)
NEG_LOG_ABAR = (-torch.log(torch.cumprod(1 - BETAS, 0))).numpy()


def matched(sigma01):
    return float(np.interp(math.log1p(4 * sigma01 ** 2), NEG_LOG_ABAR, np.arange(1000))), 1 / (1 + 4 * sigma01 ** 2)


def embed(y, ab):
    h, w = y.shape[-2:]
    yp = F.pad(y, (0, (-w) % 64, 0, (-h) % 64), mode="reflect") if (h % 64 or w % 64) else y
    return math.sqrt(ab) * (2 * yp - 1), (h, w)


@torch.no_grad()
def tweedie(model, x, t, ab, hw, classes=None):
    """Posterior means from embedded states x [B,3,H,W] (one per class label when classes is given)."""
    tt = torch.full((x.shape[0],), t, device=x.device)
    kwargs = {} if classes is None else {"y": torch.as_tensor(classes, device=x.device)}
    eps = model(x, tt, **kwargs)[:, :3].float()
    return (((x - math.sqrt(1 - ab) * eps) / math.sqrt(ab) + 1) / 2)[..., :hw[0], :hw[1]]


@torch.no_grad()
def class_probs(classifier, x, t, size):
    """p(c | x_t), averaged over the centre and four corner crops of side `size` (edge-padded if smaller)."""
    H, W = x.shape[-2:]
    if H < size or W < size:
        x = F.pad(x, (0, max(0, size - W), 0, max(0, size - H)), mode="replicate")
        H, W = x.shape[-2:]
    starts = {(0, 0), (0, W - size), (H - size, 0), (H - size, W - size), ((H - size) // 2, (W - size) // 2)}
    crops = torch.cat([x[..., i:i + size, j:j + size] for i, j in sorted(starts)])
    logits = classifier(crops, torch.full((crops.shape[0],), t, device=x.device)).float()
    return logits.softmax(-1).mean(0)


def mixture(model, x, t, ab, hw, probs):
    top = probs.topk(TOP_K)
    outs = tweedie(model, x.expand(TOP_K, -1, -1, -1), t, ab, hw, classes=top.indices)
    w = (top.values / top.values.sum()).view(-1, 1, 1, 1)
    return outs[:1], (w * outs).sum(0, keepdim=True), top


_g = torch.exp(-torch.arange(-5, 6, dtype=torch.float64) ** 2 / (2 * 1.5 ** 2))
SSIM_WINDOW = (_g / _g.sum())[:, None] * (_g / _g.sum())[None, :]


def ssim(a, b):
    """skimage structural_similarity(gaussian_weights=True, sigma=1.5, use_sample_covariance=False, data_range=255)."""
    w = SSIM_WINDOW.to(a.device)[None, None].repeat(3, 1, 1, 1)
    conv = lambda x: F.conv2d(x[None], w, groups=3)[0]
    ux, uy = conv(a), conv(b)
    vx, vy, vxy = conv(a * a) - ux * ux, conv(b * b) - uy * uy, conv(a * b) - ux * uy
    c1, c2 = (0.01 * 255) ** 2, (0.03 * 255) ** 2
    return float((((2 * ux * uy + c1) * (2 * vxy + c2)) / ((ux * ux + uy * uy + c1) * (vx + vy + c2))).mean())


def scores(x, ref_u8, lp):
    out = (x[0].clamp(0, 1) * 255).round()
    a, b = out.double(), ref_u8.double()
    mse = (a - b).pow(2).mean().item()
    with torch.no_grad():
        lpv = lp(out.float()[None] / 127.5 - 1, ref_u8.float()[None] / 127.5 - 1).item()
    return dict(psnr=10 * math.log10(255 ** 2 / mse), ssim=ssim(a, b), lpips=lpv)

## 3 · fp16 check of the 512×512 model (OpenAI sample it in fp32)

In [ ]:
with torch.no_grad():
    probe = torch.rand(1, 3, 512, 512, device=DEVICES[0])
    for s in SIGMAS:
        x, hw = embed(probe, matched(s / 255)[1])
        out = tweedie(CTX[0]["cond512"], x, matched(s / 255)[0], matched(s / 255)[1], hw, classes=[1])
        assert torch.isfinite(out).all(), f"512×512 model is not finite in fp16 at σ = {s}"
print("512×512 conditional model finite in fp16")

## 4 · Benchmarks

In [ ]:
TEST = {}
with zipfile.ZipFile(P["datasets"]) as z:
    for ds in DATASETS:
        names = sorted([n for n in z.namelist() if n.startswith(f"Datasets/Gaussion/{ds}/") and not n.endswith("/")], key=natkey)
        TEST[ds] = [(n.split("/")[-1], np.asarray(Image.open(io.BytesIO(z.read(n))).convert("RGB"))) for n in names]
if TEST_MODE:
    TEST = {ds: [(n, im[:96, :128].copy()) for n, im in v[:2]] for ds, v in TEST.items()}


def noise_seed(ds, idx, sigma):
    return 1_000_000 * (DATASETS.index(ds) + 1) + 1000 * sigma + idx


def noisy_of(clean_u8, sigma, seed):
    """DMID's protocol: Gaussian noise normalised to sample std exactly sigma/255, added in float, never clipped."""
    x = torch.from_numpy(clean_u8).permute(2, 0, 1).float() / 255
    n = torch.randn(x.shape, generator=torch.Generator().manual_seed(seed))
    return x + (sigma / 255) * n / n.std(unbiased=False)


def run_image(ctx, job):
    ds, idx, sigma = job
    name, clean_u8 = TEST[ds][idx]
    y = noisy_of(clean_u8, sigma, noise_seed(ds, idx, sigma))[None].to(ctx["dev"])
    ref = torch.from_numpy(clean_u8).permute(2, 0, 1).to(ctx["dev"])
    t, ab = matched(sigma / 255)
    x, hw = embed(y, ab)
    xf, _ = embed(y.flip(-1), ab)
    u = tweedie(ctx["uncond"], torch.cat([x, xf]), t, ab, hw)
    est = {"U ×1": u[:1], "U ×2": (u[:1] + u[1:].flip(-1)) / 2}
    p256, p512 = class_probs(ctx["cls256"], x, t, 256), class_probs(ctx["cls512"], x, t, 512)
    est["C256 top-1"], est["C256 mix"], top256 = mixture(ctx["cond256"], x, t, ab, hw, p256)
    est["C512 top-1"], est["C512 mix"], top512 = mixture(ctx["cond512"], x, t, ab, hw, p512)
    if "ft" in ctx:
        est["FT ×1"] = tweedie(ctx["ft"], x, t, ab, hw)
    est["U + C256"] = (est["U ×1"] + est["C256 mix"]) / 2
    est["U + C512"] = (est["U ×1"] + est["C512 mix"]) / 2
    est["U + C256 + C512"] = (est["U ×1"] + est["C256 mix"] + est["C512 mix"]) / 3
    est["U ×2 + C256 + C512"] = (est["U ×2"] + est["C256 mix"] + est["C512 mix"]) / 3
    if "ft" in ctx:
        est["U ×2 + C256 + C512 + FT"] = (est["U ×2"] + est["C256 mix"] + est["C512 mix"] + est["FT ×1"]) / 4
    rows = [dict(dataset=ds, image=name, sigma=sigma, method=m, **scores(o, ref, ctx["lpips"])) for m, o in est.items()]
    info = dict(dataset=ds, image=name, sigma=sigma, top256=top256.indices.tolist(), p256=top256.values.tolist(),
                top512=top512.indices.tolist(), p512=top512.values.tolist())
    return rows, info


def worker_pool(jobs, fn):
    """Run fn(ctx, job) over jobs with one thread per GPU; returns (results, errors)."""
    q, results, errors, lock = queue.Queue(), [], [], threading.Lock()
    for j in jobs:
        q.put(j)

    def loop(ctx):
        while time.time() - T_START < TIME_LIMIT_H * 3600:
            try:
                job = q.get_nowait()
            except queue.Empty:
                return
            try:
                out = fn(ctx, job)
                with lock:
                    results.append(out)
            except Exception:
                with lock:
                    errors.append((job, traceback.format_exc()))

    threads = [threading.Thread(target=loop, args=(c,)) for c in CTX]
    for th in threads:
        th.start()
    t0, last = time.time(), time.time()
    while any(th.is_alive() for th in threads):
        time.sleep(0.5 if TEST_MODE else 20)
        if time.time() - last > 600:
            last = time.time()
            print(f"  {len(results)}/{len(jobs)} done, {len(errors)} errors, {(time.time() - t0) / 60:.1f} min", flush=True)
    for job, tb in errors[:3]:
        print("ERROR", job, tb)
    return results, errors


t0 = time.time()
res, err = worker_pool([(ds, i, s) for ds in DATASETS for s in SIGMAS for i in range(len(TEST[ds]))], run_image)
SCORES = pd.DataFrame([r for rows, _ in res for r in rows])
CLASSES = pd.DataFrame([info for _, info in res])
SCORES.to_csv(f"{OUT}/scores.csv", index=False)
CLASSES.to_json(f"{OUT}/classes.json", orient="records", indent=1)
TIMES["benchmarks"] = time.time() - t0
print(f"{len(res)} images, {len(err)} errors, {TIMES['benchmarks'] / 60:.1f} min")

## 5 · Results

In [ ]:
for metric in ("psnr", "ssim", "lpips"):
    print(f"\nmean {metric.upper()}")
    print(SCORES.pivot_table(index="method", columns=["dataset", "sigma"], values=metric).round(4 if metric != "psnr" else 3).to_string())
wide = SCORES.pivot_table(index=["dataset", "sigma", "image"], columns="method", values="psnr")
print("\nPSNR gain over U ×2, pooled per σ (dB), images better / n")
for m in [c for c in wide.columns if c != "U ×2"]:
    d = (wide[m] - wide["U ×2"]).groupby("sigma")
    print(f"{m:28s}", "  ".join(f"σ{s}: {g.mean():+.4f} ({int((g > 0).sum())}/{g.size})" for s, g in d))
print("\nclassifier confidence: mean top-1 probability", CLASSES.p256.map(lambda p: p[0]).mean().round(3), CLASSES.p512.map(lambda p: p[0]).mean().round(3))
with open(f"{OUT}/summary.json", "w") as f:
    json.dump(dict(minutes={k: v / 60 for k, v in TIMES.items()}, fine_tuned=FT_PATH, top_k=TOP_K,
                   mean=SCORES.groupby(["dataset", "sigma", "method"])[["psnr", "ssim", "lpips"]].mean().reset_index().to_dict("records")),
              f, indent=1, default=float)
open(f"{OUT}/DONE", "w").write(time.strftime("%Y-%m-%d %H:%M:%S"))
print(sorted(os.listdir(OUT)), f"total {(time.time() - T_START) / 60:.1f} min")